## Example 1: Basic tracing API and span operations

In [ ]:
# Example 1: Basic tracing API and span operations
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode, SpanKind
import time

print("Example 1 - Basic tracing API and span operations")

try:
    # Get the default tracer with service information
    tracer = trace.get_tracer(__name__, instrumenting_library_version="1.0.0")
    print(f"✓ Tracer created: {type(tracer).__name__}")
    
    # Test tracer provider access
    provider = trace.get_tracer_provider()
    print(f"  Tracer provider: {type(provider).__name__}")
    
    # Create a span with comprehensive attributes
    with tracer.start_as_current_span("user_operation", kind=SpanKind.SERVER) as span:
        print("  Inside span context")
        
        # Set various attribute types
        span.set_attribute("service.name", "example_service")
        span.set_attribute("operation.type", "user_request")
        span.set_attribute("user.id", 12345)
        
        span_context = span.get_span_context()
        print(f"  Span ID: {format(span_context.span_id, '016x')}")
        print(f"  Trace ID: {format(span_context.trace_id, '032x')}")
        
        # Add events
        span.add_event("request_received")
        time.sleep(0.01)
        span.add_event("processing_completed")
        
        # Set successful status
        span.set_status(Status(StatusCode.OK))
        print("  ✓ Span completed successfully")
    
    print("Example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_api example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Nested spans and context propagation

In [ ]:
# Example 2: Nested spans and context propagation  
from opentelemetry import trace
from opentelemetry.trace import SpanKind, Status, StatusCode
import time

print("Example 2 - Nested spans and context propagation")

try:
    # Create tracer
    tracer = trace.get_tracer("distributed_service", instrumenting_library_version="2.1.0")
    
    # Create parent span
    with tracer.start_as_current_span("handle_request", kind=SpanKind.SERVER) as root_span:
        root_span.set_attribute("http.method", "POST")
        root_span.set_attribute("service.name", "user_service")
        
        print(f"✓ Root span created")
        
        # Child span 1: Database operation
        with tracer.start_as_current_span("database_query", kind=SpanKind.CLIENT) as db_span:
            db_span.set_attribute("db.system", "postgresql")
            db_span.set_attribute("db.operation", "select")
            time.sleep(0.01)
            print("  ✓ Database operation completed")
        
        # Child span 2: External service call
        with tracer.start_as_current_span("auth_service", kind=SpanKind.CLIENT) as auth_span:
            auth_span.set_attribute("http.url", "https://auth.example.com")
            time.sleep(0.01)
            print("  ✓ External service call completed")
        
        root_span.set_status(Status(StatusCode.OK))
    
    print("Example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_api example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Metrics API with comprehensive instrument types

In [ ]:
# Example 3: Metrics API with comprehensive instrument types
from opentelemetry import metrics
from opentelemetry.metrics import CallbackOptions, Observation
import time
import random
from typing import Iterable

print("Example 3 - Metrics API with comprehensive instrument types")

try:
    # Create meter with detailed information
    meter = metrics.get_meter(
        "example_service_metrics",
        version="1.0.0",
        schema_url="https://opentelemetry.io/schemas/1.21.0"
    )
    print(f"✓ Meter created: {type(meter).__name__}")
    
    # Test meter provider access
    meter_provider = metrics.get_meter_provider()
    print(f"  Meter provider: {type(meter_provider).__name__}")
    
    # 1. Counter - Monotonically increasing values
    request_counter = meter.create_counter(
        name="http_requests_total",
        description="Total number of HTTP requests received",
        unit="1"
    )
    
    error_counter = meter.create_counter(
        name="http_errors_total",
        description="Total number of HTTP errors",
        unit="1"
    )
    
    # 2. UpDownCounter - Values that can increase or decrease
    active_connections = meter.create_up_down_counter(
        name="active_connections",
        description="Current number of active connections",
        unit="1"
    )
    
    queue_size = meter.create_up_down_counter(
        name="queue_size",
        description="Current size of the processing queue",
        unit="1"
    )
    
    # 3. Histogram - Distribution of values
    request_duration = meter.create_histogram(
        name="http_request_duration_seconds",
        description="Duration of HTTP requests in seconds",
        unit="s"
    )
    
    response_size = meter.create_histogram(
        name="http_response_size_bytes",
        description="Size of HTTP responses in bytes",
        unit="By"
    )
    
    # 4. Observable Gauge - Current state via callback
    def get_system_metrics(options: CallbackOptions) -> Iterable[Observation]:
        """Callback function to provide current system metrics"""
        # Simulate system metrics readings
        yield Observation(random.uniform(0, 100), {"metric": "cpu_usage", "core": "0"})
        yield Observation(random.uniform(0, 100), {"metric": "cpu_usage", "core": "1"})
        yield Observation(random.uniform(0, 8192), {"metric": "memory_usage", "type": "heap"})
        yield Observation(random.uniform(0, 4096), {"metric": "memory_usage", "type": "stack"})
        yield Observation(random.uniform(0, 100), {"metric": "disk_usage", "mount": "/"})
        yield Observation(random.uniform(0, 100), {"metric": "disk_usage", "mount": "/data"})
    
    system_gauge = meter.create_observable_gauge(
        name="system_metrics",
        description="Current system resource usage",
        unit="1",
        callbacks=[get_system_metrics]
    )
    
    # 5. Observable Counter - Monotonic values via callback
    def get_cumulative_metrics(options: CallbackOptions) -> Iterable[Observation]:
        """Callback for cumulative metrics"""
        # Simulate cumulative counters
        yield Observation(time.time() * 1000, {"metric": "uptime_seconds"})
        yield Observation(random.randint(1000, 10000), {"metric": "total_bytes_processed"})
        yield Observation(random.randint(100, 1000), {"metric": "total_operations"})
    
    cumulative_counter = meter.create_observable_counter(
        name="cumulative_metrics",
        description="Cumulative system metrics",
        unit="1",
        callbacks=[get_cumulative_metrics]
    )
    
    # 6. Observable UpDownCounter - Current values that can increase/decrease
    def get_current_state(options: CallbackOptions) -> Iterable[Observation]:
        """Callback for current state metrics"""
        yield Observation(random.randint(-100, 100), {"metric": "balance", "account": "checking"})
        yield Observation(random.randint(-50, 200), {"metric": "balance", "account": "savings"})
        yield Observation(random.randint(0, 1000), {"metric": "inventory", "product": "widget_a"})
        yield Observation(random.randint(0, 500), {"metric": "inventory", "product": "widget_b"})
    
    state_counter = meter.create_observable_up_down_counter(
        name="current_state_metrics",
        description="Current state values that can increase or decrease",
        unit="1",
        callbacks=[get_current_state]
    )
    
    print(f"✓ Created {6} metric instruments:")
    print(f"  Counter: {request_counter._name}")
    print(f"  UpDownCounter: {active_connections._name}")
    print(f"  Histogram: {request_duration._name}")
    print(f"  Observable Gauge: {system_gauge._name}")
    print(f"  Observable Counter: {cumulative_counter._name}")
    print(f"  Observable UpDownCounter: {state_counter._name}")
    
    # Record comprehensive measurements
    print(f"\n✓ Recording measurements:")
    
    # Simulate HTTP request processing
    for i in range(10):
        method = random.choice(["GET", "POST", "PUT", "DELETE"])
        status = random.choice([200, 201, 400, 404, 500])
        endpoint = random.choice(["/api/users", "/api/orders", "/api/products"])
        
        # Record request
        request_counter.add(1, {
            "method": method,
            "endpoint": endpoint,
            "status": str(status)
        })
        
        # Record error if applicable
        if status >= 400:
            error_counter.add(1, {
                "method": method,
                "endpoint": endpoint,
                "status": str(status),
                "error_type": "client_error" if status < 500 else "server_error"
            })
        
        # Record request duration
        duration = random.uniform(0.01, 2.0)
        request_duration.record(duration, {
            "method": method,
            "endpoint": endpoint,
            "status": str(status)
        })
        
        # Record response size
        size = random.randint(100, 10000)
        response_size.record(size, {
            "method": method,
            "content_type": "application/json"
        })
        
        print(f"  Request {i+1}: {method} {endpoint} -> {status} ({duration:.3f}s, {size}B)")
    
    # Record connection changes
    print(f"\n✓ Recording connection changes:")
    active_connections.add(5, {"server": "web-01"})
    active_connections.add(3, {"server": "web-02"})
    print(f"  Added 8 total connections")
    
    time.sleep(0.1)  # Simulate some time passing
    
    active_connections.add(-2, {"server": "web-01"})
    active_connections.add(-1, {"server": "web-02"})
    print(f"  Removed 3 connections")
    
    # Record queue operations
    print(f"\n✓ Recording queue operations:")
    queue_size.add(20, {"queue": "processing"})
    queue_size.add(15, {"queue": "notification"})
    print(f"  Added items to queues")
    
    time.sleep(0.05)
    
    queue_size.add(-10, {"queue": "processing"})
    queue_size.add(-5, {"queue": "notification"})
    print(f"  Processed items from queues")
    
    # Test metric attributes and validation
    print(f"\n✓ Testing metric attributes:")
    
    # Test various attribute types
    test_attributes = {
        "string_attr": "test_value",
        "int_attr": 42,
        "float_attr": 3.14,
        "bool_attr": True
    }
    
    for attr_name, attr_value in test_attributes.items():
        try:
            request_counter.add(1, {attr_name: attr_value, "test_type": "attribute_validation"})
            print(f"  ✓ {attr_name}: {type(attr_value).__name__}")
        except Exception as e:
            print(f"  ✗ {attr_name}: {type(attr_value).__name__} - Error: {e}")
            raise
    
    # Observable metrics are automatically collected via callbacks
    print(f"\n✓ Observable metrics configured with callbacks")
    print(f"  System metrics: CPU, memory, disk usage")
    print(f"  Cumulative metrics: Uptime, total bytes, operations")
    print(f"  State metrics: Account balances, inventory levels")
    
    print("\nMetrics recording completed successfully")
    print("Note: Actual metric collection depends on configured metric providers/exporters")
    
    print("Example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_api example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Context management and baggage

In [ ]:
# Example 4: Context management and baggage
from opentelemetry import baggage, context
from opentelemetry.baggage import set_baggage, get_baggage, get_all, remove_baggage
import time

print("Example 4 - Context management and baggage")

try:
    # Test baggage operations
    print("✓ Testing baggage operations:")
    
    # Set baggage values
    ctx = context.get_current()
    ctx = set_baggage("user.id", "user_123", context=ctx)
    ctx = set_baggage("session.id", "session_abc", context=ctx)
    ctx = set_baggage("environment", "production", context=ctx)
    
    print(f"  ✓ Set 3 baggage items")
    
    # Access baggage with context
    token = context.attach(ctx)
    try:
        user_id = get_baggage("user.id")
        session_id = get_baggage("session.id")
        environment = get_baggage("environment")
        
        print(f"  User ID: {user_id}")
        print(f"  Session ID: {session_id}")
        print(f"  Environment: {environment}")
        assert (user_id, session_id, environment) == ("user_123", "session_abc", "production")
        
        # Get all baggage
        all_baggage = get_all()
        print(f"  Total baggage items: {len(all_baggage)}")
        assert dict(all_baggage) == {"user.id": "user_123", "session.id": "session_abc", "environment": "production"}
        
    finally:
        context.detach(token)
    # After detach the baggage is no longer in the current context
    assert get_baggage("user.id") is None
    
    # Test baggage removal
    ctx = remove_baggage("environment", context=ctx)
    token = context.attach(ctx)
    try:
        removed = get_baggage("environment")
        print(f"  Removed environment: {removed}")
        assert removed is None
        assert get_baggage("user.id") == "user_123"
    finally:
        context.detach(token)
    
    print("Example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_api example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Exception recording and error handling

In [ ]:
# Example 5: Exception recording and error handling
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode, SpanKind
import time

print("Example 5 - Exception recording and error handling")

try:
    tracer = trace.get_tracer("error_handling_test")
    
    # Test exception recording
    with tracer.start_as_current_span("error_handling_span") as error_span:
        try:
            error_span.set_attribute("operation.type", "error_simulation")
            time.sleep(0.01)
            raise ValueError("Invalid input data")
            
        except ValueError as ve:
            # Record the exception
            error_span.record_exception(ve)
            error_span.set_status(Status(StatusCode.ERROR, str(ve)))
            print(f"  ✓ Recorded exception: {ve}")
    
    # Test successful span
    with tracer.start_as_current_span("successful_span") as success_span:
        success_span.set_attribute("operation.type", "success")
        success_span.set_status(Status(StatusCode.OK))
        print(f"  ✓ Successful operation completed")
    
    print("Example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_api example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")